In [1]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from lsst.ts.observatory.control.utils import RotType

In [2]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [3]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [4]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Could not read historical data in 60.11 sec
Read 62 history items for RemoteEvent(ATHeaderService, 0, heartbeat)
Could not read historical data in 60.13 sec
Read 72 history items for RemoteEvent(ATDomeTrajectory, 0, heartbeat)
Could not read historical data in 60.14 sec
Read 61 history items for RemoteEvent(ATArchiver, 0, heartbeat)
Could not read historical data in 60.19 sec
Read 77 history items for RemoteEvent(ATHexapod, 0, heartbeat)
positionStatus DDS read queue is filling: 75 of 100 elements
Could not read historical data in 60.23 sec
Read 63 history items for RemoteEvent(ATSpectrograph, 0, heartbeat)
Could not read h

[[None, None, None, None, None, None, None], [None, None, None, None]]

currentTargetStatus DDS read queue is full (100 elements); data may be lost
nasymth_m3_mountMotorEncoders DDS read queue is filling: 93 of 100 elements
mount_Nasmyth_Encoders DDS read queue is filling: 93 of 100 elements
mount_AzEl_Encoders DDS read queue is filling: 93 of 100 elements
mount_AzEl_Encoders python read queue is filling: 92 of 100 elements
measuredTorque DDS read queue is filling: 93 of 100 elements
measuredMotorVelocity DDS read queue is filling: 93 of 100 elements
azEl_mountMotorEncoders DDS read queue is filling: 95 of 100 elements


In [ ]:
# enable components
await atcs.enable({"atdome": "current", "ataos": "current", "athexapod": "current"})
await latiss.enable()

In [ ]:
await latiss.take_bias(5)
# Forgot to add wait between them.

In [ ]:
# Take 5 biases
# Added wait to stop killing the recent images
for i in range(5):
    await asyncio.sleep(2.0)
    await latiss.take_bias(1)

In [ ]:
# Take 5 10 second darks 
await latiss.take_darks(10.0, 5)

In [ ]:
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [ ]:
#await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply='current')

In [ ]:
#await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED, settingsToApply='current')

In [ ]:
await atcs.prepare_for_flatfield()

In [ ]:
await atcs.rem.ataos.cmd_enableCorrection.set_start(
    m1=True, hexapod=True, atspectrograph=True, timeout=atcs.long_timeout)

In [ ]:
await latiss.setup_atspec(filter='RG610')

In [ ]:
# Take 2 2 second flats 
await latiss.take_flats(2.0, 2, filter='RG610', grating='empty_1')

In [ ]:
# Take 2 2 second flats 
await latiss.take_flats(2.0, 2, filter='RG610', grating='empty_1')

In [ ]:
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [ ]:
#await atcs.prepare_for_onsky()

In [ ]:
# Opens/Closes dropout shutter
#await atcs.rem.atdome.cmd_moveShutterDropoutDoor.set_start(open=False)

In [ ]:
# Don't forget the 'settingsToApply='current' or it won't work!!
#await salobj.set_summary_state(atcs.rem.ataos, salobj.State.ENABLED, settingsToApply='current')

In [ ]:
#await atcs.rem.ataos.cmd_enableCorrection.set_start(
#    m1=True, hexapod=True, atspectrograph=True, timeout=atcs.long_timeout
#)

In [ ]:
# HD72514 Az~120, Alt~65 m=8.05

In [ ]:
current_az = 200.0
current_el = 50.0
current_rot = 0.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

In [ ]:
#await salobj.set_summary_state(atcs.rem.atptg, salobj.State.ENABLED)

In [ ]:
#await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.ENABLED)

In [ ]:
#await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED)

In [ ]:
#await latiss.setup_instrument()

In [ ]:
#await latiss.setup_instrument

In [ ]:
#await latiss.setup_atspec(filter='RG610')

In [ ]:
atcs.rem.atptg.cmd_raDecTarget.set(azWrapStrategy=1) 

In [ ]:
# HD26491 m = 6.35, Az = +51, Alt = +200
# HD177482 near pole
#await atcs.slew_object('HD 177482', rot=0.0, rot_type=RotType.PhysicalSky)
await atcs.slew_object('HD 26491', rot=45.0, rot_type=RotType.PhysicalSky)

In [ ]:
await latiss.take_object(5.0, 1, filter='RG610', grating='empty_1')

In [ ]:
# Move the star to the bottom of the field
await atcs.offset_xy(y=140, x=0)

In [ ]:
await latiss.setup_instrument(filter='RG610', grating='ronchi90lpmm')

In [ ]:
await latiss.take_object(5.0, 1, filter='RG610', grating='ronchi90lpmm')

In [ ]:
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [ ]:
await atcs.shutdown()

In [ ]:
# Putting everything back in standby.
await latiss.standby()